Imports and Model Loading

In [1]:
import pandas as pd
import numpy as np
import sys
import time
from sentence_transformers import CrossEncoder

sys.path.append("..")
from src.retrieval.hybrid_engine import HybridRetrievalEngine

print("Loading CrossEncoder model ('cross-encoder/ms-marco-MiniLM-L-6-v2')...")
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-Encoder loaded successfully.")

# Initialize our Phase 3 Hybrid Retrieval Engine
hybrid_engine = HybridRetrievalEngine(
    data_path="../data/books_cleaned.csv",
    chroma_dir="../data/chroma_db"
)


/Users/simranchaudhary/Documents/ShelfKnows/venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/simranchaudhary/Documents/ShelfKnows/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading CrossEncoder model ('cross-encoder/ms-marco-MiniLM-L-6-v2')...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded successfully.
Initializing Hybrid Retrieval Engines (Vector + BM25 + Collaborative Filtering)...
Initializing Embedding Model ('all-MiniLM-L6-v2')...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Connecting to local ChromaDB vector store at '../data/chroma_db'...
 Building BM25 Sparse Index from '../data/books_cleaned.csv'...
 BM25 Index successfully built for 50000 books in 0.84 seconds!
 Initializing Collaborative Filtering Engine from '../data/books_cleaned.csv'...
 Collaborative Filtering Model trained across 500 books in 1.46 seconds!
All three engines loaded successfully.


Stage 1 Retrieval (Get Top 15 Candidates)

In [2]:
query = "books about high-stakes survival games and rebellion against oppressive regimes"

print(f"Executing Stage 1 Hybrid Retrieval for: '{query}'")
candidates = hybrid_engine.search(query, top_k=15)

print(f"Retrieved {len(candidates)} candidates from Stage 1.")
df_stage1 = pd.DataFrame(candidates)[['title', 'authors', 'genres', 'average_rating', 'rrf_score']]
display(df_stage1.head(5))


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Executing Stage 1 Hybrid Retrieval for: 'books about high-stakes survival games and rebellion against oppressive regimes'
⚡ Vector Search for 'books about high-stakes survival games and rebellion against oppressive regimes' returned 30 matches in 186.17 ms
⚡ BM25 Search for 'books about high-stakes survival games and rebellion against oppressive regimes' returned 30 matches in 104.38 ms
 Book 'Seeds of Rebellion (Beyonders, #2)' not found in top interaction catalog.
Hybrid Retrieval completed for 'books about high-stakes survival games and rebellion against oppressive regimes' in 295.84 ms
Retrieved 15 candidates from Stage 1.


,title,authors,genres,average_rating,rrf_score
0,"Seeds of Rebellion (Beyonders, #2)",Brandon Mull,General Fiction,4.25,0.03175
1,"Unbroken: A World War II Story of Survival, Re...",Laura Hillenbrand,History & Biography,4.40,0.01639
2,The Most Dangerous Game,Richard Connell,General Fiction,3.91,0.01587
3,Defiance (Defiance #1),C.J. Redwine,General Fiction,3.69,0.01562
4,The Westing Game,Ellen Raskin,General Fiction,4.03,0.01538


Stage 2 Neural Cross-Encoder Scoring

In [4]:
print("Constructing Query-Book sentence pairs for Cross-Encoder scoring...")

# Prepare (Query, Document) pairs safely using .get()
pairs = []
for book in candidates:
    # Use description if present, otherwise construct a descriptive summary from metadata
    desc = book.get('description') or f"A renowned {book.get('genres', 'Fiction')} book titled '{book.get('title', '')}' by {book.get('authors', '')}."
    text_representation = f"Title: {book.get('title', '')} | Genre: {book.get('genres', '')} | Summary: {desc}"
    pairs.append([query, text_representation])

start_time = time.time()
# Predict deep relevance logits using the neural Cross-Encoder
cross_scores = reranker_model.predict(pairs)
elapsed_ms = (time.time() - start_time) * 1000

print(f"Cross-Encoder scored {len(candidates)} candidate pairs in {elapsed_ms:.2f} ms.")

# Attach neural reranker scores to candidates
for idx, book in enumerate(candidates):
    book['cross_encoder_score'] = round(float(cross_scores[idx]), 4)

# Sort candidates by Cross-Encoder score in descending order
reranked_candidates = sorted(candidates, key=lambda b: b['cross_encoder_score'], reverse=True)
print("Reranking complete!")


Constructing Query-Book sentence pairs for Cross-Encoder scoring...
Cross-Encoder scored 15 candidate pairs in 905.11 ms.
Reranking complete!


Side-by-Side Comparison: Before vs. After Reranking

In [5]:
print("BEFORE Reranking (Stage 1 - Hybrid Retrieval Order):")
df_before = pd.DataFrame(candidates)[['title', 'authors', 'genres', 'rrf_score']].head(5)
display(df_before)

print("\nAFTER Reranking (Stage 2 - Cross-Encoder Neural Re-ordering):")
df_after = pd.DataFrame(reranked_candidates)[['title', 'authors', 'genres', 'cross_encoder_score']].head(5)
display(df_after)


BEFORE Reranking (Stage 1 - Hybrid Retrieval Order):


,title,authors,genres,rrf_score
0,"Seeds of Rebellion (Beyonders, #2)",Brandon Mull,General Fiction,0.03175
1,"Unbroken: A World War II Story of Survival, Re...",Laura Hillenbrand,History & Biography,0.01639
2,The Most Dangerous Game,Richard Connell,General Fiction,0.01587
3,Defiance (Defiance #1),C.J. Redwine,General Fiction,0.01562
4,The Westing Game,Ellen Raskin,General Fiction,0.01538



AFTER Reranking (Stage 2 - Cross-Encoder Neural Re-ordering):


,title,authors,genres,cross_encoder_score
0,"Seeds of Rebellion (Beyonders, #2)",Brandon Mull,General Fiction,-4.6908
1,The Most Dangerous Game,Richard Connell,General Fiction,-7.6459
2,"Unbroken: A World War II Story of Survival, Re...",Laura Hillenbrand,History & Biography,-7.6632
3,The Westing Game,Ellen Raskin,General Fiction,-8.2767
4,"The Egypt Game (Game, #1)",Zilpha Keatley Snyder,General Fiction,-8.3962
